# Object-level evaluation on Traffic Signs (AE point 1)

Table 1 scores whole 256 px crops. This notebook scores **the object region only**: every sample set carries, in its `meta.json`, the box
each image was conditioned on and the name of the real training image that box came from. For each set it
1. cuts a square crop around the box out of the generated image and out of its real counterpart (same box), 3,000 pairs,
   and scores the two crop sets with dgm-eval (DINOv2 FD, KD, precision, recall; Inception FID);
2. runs a COCO-pretrained person detector (YOLO11n, class *person*) on the full generated images and reports how often a person is
   found at all, how often one overlaps the conditioned box with IoU ≥ 0.5, and the mean best IoU; the same on the real images gives the reference.

One Excel row per set, appended after every set, so the cell can be re-run after a disconnect. Same install cells as Step14.

**Run order:** cells 1–2, package pins → **Runtime ▸ Restart session** → version check, mount, detector install, real images, `%cd`, environment, CONFIG.


In [ ]:
%cd /content
!rm -rf dgm-eval
!git clone https://github.com/layer6ai-labs/dgm-eval.git
%cd dgm-eval


In [ ]:
!pip install --quiet --no-deps -e .


**Restart Session after the cell below** (Runtime ▸ Restart session), then continue from the version-check cell.


In [ ]:
# Core scientific stack (plays well on Colab Py3.12)
!pip install -q "numpy==2.2.6" "scipy==1.16.1" "pandas==2.2.2" "scikit-learn==1.7.2"

# Vision + image utils
!pip install -q "opencv-python==4.12.0.88" "pillow==10.4.0" "scikit-image==0.24.0"

# Encoders used by dgm-eval  (xformers deliberately NOT installed any more: the Oct 2026 Colab image ships
# PyTorch 2.11 / Python 3.13, the xformers wheel was built for PyTorch 2.10 / Python 3.10, and DINOv2 then crashes
# inside memory_efficient_attention. DINOv2 runs its plain attention instead; the features are the same.)
!pip install -q "open_clip_torch==2.19.0" "timm==0.9.16" "transformers==4.45.2"


In [ ]:
import numpy, pandas, sklearn, scipy, PIL, cv2, skimage
import torch, torchvision
import open_clip, timm, transformers

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("SciPy:", scipy.__version__)
print("Pillow:", PIL.__version__)
print("cv2:", cv2.__version__)
print("skimage:", skimage.__version__)
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("open_clip:", getattr(open_clip, "__version__", "ok"))
print("timm:", timm.__version__)
print("transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")
print("All good.")


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# create symbolic link
!ln -s /content/gdrive/My\ Drive/ /mydrive

# list gdrive content
!ls /mydrive


In [ ]:
# the person detector; --no-deps keeps the pinned stack untouched (Colab already has torch, opencv, pandas, matplotlib, tqdm, pyyaml)
!pip install -q --no-deps "ultralytics==8.3.158" "ultralytics-thop" "py-cpuinfo"
from ultralytics import YOLO
print("ultralytics OK")
# if the import above fails, run instead:  !pip install -q ultralytics

## Real images

The real training images; each generated sample names the one its box came from, so no box files are needed. The COCO-pretrained detector knows only one traffic-sign class (stop sign, class 11), while the dataset holds many sign types. The detector columns are therefore a weak placement indicator here and must be read against the real-image reference row; the crop FD/KD columns do not depend on the detector.

In [ ]:
REAL_ZIP = "/content/gdrive/MyDrive/PhD/TrafficSigns/TrafficSigns_img.zip"
import os, zipfile
REAL_DIR = "/content/dgm-eval/TrafficSigns_img_flat"
if not (os.path.isdir(REAL_DIR) and os.listdir(REAL_DIR)):
    os.makedirs(REAL_DIR, exist_ok=True)
    with zipfile.ZipFile(REAL_ZIP) as z:
        for n in z.namelist():
            if n.lower().endswith((".png", ".jpg", ".jpeg")) and not os.path.basename(n).startswith("._"):
                with open(os.path.join(REAL_DIR, os.path.basename(n)), "wb") as f:
                    f.write(z.read(n))
REAL_INDEX = {os.path.splitext(f)[0]: os.path.join(REAL_DIR, f) for f in os.listdir(REAL_DIR)}
print("real images:", len(REAL_INDEX))

In [ ]:
# ==== keep TensorFlow out of the way ====================================================================
# open_clip_torch 2.19.0 pins protobuf<4, and Colab's TensorFlow (Python 3.13 image) needs protobuf>=5.26.
# `transformers` imports TensorFlow whenever it sees it installed, which is where the
# "cannot import name 'runtime_version' from 'google.protobuf'" crash came from. dgm-eval never uses TensorFlow,
# so we tell `transformers` to ignore it. The two variables are inherited by every !python command below.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
# XFORMERS_DISABLED=1 is read by the DINOv2 hub code (dinov2/layers/attention.py, swiglu_ffn.py, block.py):
# with it set, DINOv2 uses its ordinary PyTorch attention instead of xformers' memory_efficient_attention,
# which cannot run when the installed xformers wheel does not match Colab's PyTorch build.
os.environ["XFORMERS_DISABLED"] = "1"

# smoke test: this must print the encoder list (inception, dinov2, ...) and no traceback
!USE_TF=0 USE_TORCH=1 XFORMERS_DISABLED=1 python -c "import google.protobuf as p; print('protobuf', p.__version__); from dgm_eval.models import MODELS; print('dgm-eval import OK:', list(MODELS))"


In [ ]:
# ==== FALLBACKS - run ONLY if the smoke test above still shows a traceback ===============================
# F1: newer protobuf (overrides open_clip's old pin; open_clip only needs protobuf for text tokenisers, not images)
#!pip install -q "protobuf>=5.29,<7"
# F2: drop the unused data2vec encoder, which is the only thing in dgm-eval that imports `transformers`
#!sed -i 's/^from .data2vec import HuggingFaceTransformerEncoder/HuggingFaceTransformerEncoder = None  # disabled/' /content/dgm-eval/dgm_eval/models/__init__.py /content/dgm-eval/dgm_eval/models/load_encoder.py
# F3: if DINOv2 still dies inside xformers "memory_efficient_attention" (an old xformers wheel is installed in this
#     session), remove xformers altogether; DINOv2 then falls back to plain attention by itself
#!pip uninstall -y -q xformers
# then re-run the smoke test cell above


## CONFIG and the loop

`INCLUDE` filters the zips by substrings of their names (empty list = every `*_n3000_*.zip` in the folder). The real-image detector
reference is computed once per notebook run. Crops are 128 px squares around the box with 10 % padding, the box being clipped to the image.

In [ ]:
# ==== object-level evaluation core (same code as tools/objeval_core.py in the package) ====
# -*- coding: utf-8 -*-
"""Object-level evaluation core (AE point 1): box crops of generated and matching real images, and person detection
inside the conditioned box with a COCO-pretrained YOLO. Shared by the local test and the Colab notebook."""
import os, json, glob
import numpy as np
from PIL import Image

PERSON_CLASS = 11         # COCO class index of "stop sign" in the Ultralytics models (target class of this dataset)
CROP_SIZE = 128           # crops are saved at this size; dgm-eval's encoders resize to their input anyway


def square_crop(im, box, pad_ratio=0.10):
    """Square crop around the box (padded by pad_ratio of the longer side), clipped to the image, resized to CROP_SIZE."""
    W, H = im.size
    x1, y1, x2, y2 = box
    side = int(round(max(x2 - x1, y2 - y1) * (1.0 + 2.0 * pad_ratio))); side = max(8, min(side, min(W, H)))
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    sx1 = min(max(0, cx - side // 2), W - side); sy1 = min(max(0, cy - side // 2), H - side)
    return im.crop((sx1, sy1, sx1 + side, sy1 + side)).resize((CROP_SIZE, CROP_SIZE), Image.LANCZOS)


def build_crop_sets(gen_dir, meta, real_index, out_gen, out_real):
    """gen_dir: flat folder of sample_*.png; meta: the set's meta.json dict; real_index: {stem: real image path}.
    Writes paired crops (same box) for the generated image and the real image the box came from."""
    os.makedirs(out_gen, exist_ok=True); os.makedirs(out_real, exist_ok=True); n = 0; missing = 0
    for s in meta["samples"]:
        gp = os.path.join(gen_dir, s["file"]); rp = real_index.get(s["bb_stem"])
        if not (os.path.exists(gp) and rp):
            missing += 1; continue
        box = [int(v) for v in s["bb_xyxy"]]
        square_crop(Image.open(gp).convert("RGB"), box).save(os.path.join(out_gen, s["file"]))
        square_crop(Image.open(rp).convert("RGB"), box).save(os.path.join(out_real, s["file"]))
        n += 1
    return n, missing


def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def detect_persons(model, image_paths, boxes, conf=0.25, batch=32):
    """Runs the detector on full images. For each image: whether any person was found anywhere, whether one overlaps the
    conditioned box with IoU >= 0.5, and the best IoU. Returns per-image records."""
    recs = []
    for i in range(0, len(image_paths), batch):
        paths = image_paths[i:i + batch]
        results = model.predict(paths, conf=conf, classes=[PERSON_CLASS], verbose=False, imgsz=256)
        for p, box, r in zip(paths, boxes[i:i + batch], results):
            dets = r.boxes.xyxy.cpu().numpy().tolist() if r.boxes is not None and len(r.boxes) else []
            best = max((iou(box, d) for d in dets), default=0.0)
            recs.append(dict(file=os.path.basename(p), n_persons=len(dets), any_person=int(len(dets) > 0), hit_iou50=int(best >= 0.5), best_iou=best))
    return recs


def summarize(recs):
    a = np.array([[r["any_person"], r["hit_iou50"], r["best_iou"]] for r in recs], dtype=float)
    return dict(n=len(recs), person_detected_pct=100 * a[:, 0].mean(), person_in_box_iou50_pct=100 * a[:, 1].mean(), mean_best_iou=float(a[:, 2].mean()))


In [ ]:
# ==== CONFIG ===============================================================================
SETS_DIR    = "/content/gdrive/MyDrive/PhD/TrafficSigns/Sets_200k"
RESULTS_DIR = "/content/gdrive/MyDrive/PhD/TrafficSigns/Sets_200k_metrics"
INCLUDE     = []            # e.g. ["anc500_cfg15", "adm_ema", "adm_mask", "nonoise", "nomask", "nocfg"]; [] = all sets
NSAMPLE     = 3000
# ============================================================================================
import re, glob, shutil, subprocess, time, pandas as pd
os.makedirs(RESULTS_DIR, exist_ok=True)
XLSX = f"{RESULTS_DIR}/Traffic_object_metrics.xlsx"
ENV = dict(os.environ, USE_TF="0", USE_TORCH="1", XFORMERS_DISABLED="1")

def parse(txt_path):
    out = {}
    for line in open(txt_path):
        m = re.match(r'^([A-Za-z_\-]+):\s*([-+0-9.eE]+)$', line.strip())
        if m:
            try: out[m.group(1)] = float(m.group(2))
            except ValueError: pass
    return out

def run_dgm_crops(model, metrics, real_crops, gen_crops, tag):
    txt = f"{RESULTS_DIR}/{tag}_crops_{model}.txt"
    cmd = ["python", "-m", "dgm_eval", real_crops, gen_crops, "--model", model, "--metrics", *metrics.split(), "--device", "cuda",
           "--batch_size", "128", "--nsample", str(NSAMPLE), "--seed", "42", "--output_dir", "/content/dgm-eval/experiments_crops_traffic"]
    with open(txt, "w") as f:
        p = subprocess.run(cmd, cwd="/content/dgm-eval", env=ENV, stdout=f, stderr=subprocess.STDOUT)
    res = parse(txt)
    if p.returncode != 0 or "fd" not in res:
        print(open(txt).read()[-2000:]); raise RuntimeError(f"dgm-eval failed on crops for {tag} / {model}: see {txt}")
    return res

detector = YOLO("yolo11n.pt")
done = set(pd.read_excel(XLSX, index_col=0).index.astype(str)) if os.path.exists(XLSX) else set()
if done: print("already scored:", sorted(done))
zips = sorted(z for z in glob.glob(f"{SETS_DIR}/*_n3000_*.zip") if not INCLUDE or any(k in os.path.basename(z) for k in INCLUDE))
assert zips, "no matching zips"
print(len(zips), "sets")
real_ref = None
for zp in zips:
    tag = os.path.basename(zp)[:-4]
    if tag in done:
        print("skip (done):", tag); continue
    t0 = time.time(); print("\n=====", tag, "=====")
    raw = f"/content/dgm-eval/_raw_{tag}"; gen = f"/content/dgm-eval/gen_{tag}"
    cg = f"/content/dgm-eval/crops_gen_{tag}"; cr = f"/content/dgm-eval/crops_real_{tag}"
    for d in (raw, gen, cg, cr): shutil.rmtree(d, ignore_errors=True)
    os.makedirs(gen)
    with zipfile.ZipFile(zp) as z: z.extractall(raw)
    meta = None
    for dp, dn, fn in os.walk(raw):
        for f in fn:
            if f == "meta.json": meta = json.load(open(os.path.join(dp, f)))
            elif f.startswith("sample_") and f.lower().endswith(".png"): shutil.copy2(os.path.join(dp, f), os.path.join(gen, f))
    shutil.rmtree(raw)
    assert meta is not None, "no meta.json in the zip"
    n_pairs, missing = build_crop_sets(gen, meta, REAL_INDEX, cg, cr)
    print(f"crop pairs {n_pairs} (missing {missing})")
    A = run_dgm_crops("dinov2", "fd kd prdc", cr, cg, tag)
    B = run_dgm_crops("inception", "fd kd", cr, cg, tag)
    boxes = [s["bb_xyxy"] for s in meta["samples"]]
    det = summarize(detect_persons(detector, [os.path.join(gen, s["file"]) for s in meta["samples"]], boxes))
    if real_ref is None:
        rp = [REAL_INDEX[s["bb_stem"]] for s in meta["samples"] if s["bb_stem"] in REAL_INDEX]
        rb = [s["bb_xyxy"] for s in meta["samples"] if s["bb_stem"] in REAL_INDEX]
        real_ref = summarize(detect_persons(detector, rp, rb)); print("real-image detector reference:", real_ref)
    row = {"crop_dinov2_" + k: v for k, v in A.items()}; row.update({"crop_inception_" + k: v for k, v in B.items()})
    row.update({"det_" + k.replace("person", "target"): v for k, v in det.items()}); row.update({"real_det_" + k.replace("person", "target"): v for k, v in real_ref.items()})
    row["crop_pairs"] = n_pairs
    table = pd.read_excel(XLSX, index_col=0) if os.path.exists(XLSX) else pd.DataFrame()
    table = pd.concat([table, pd.DataFrame([row], index=[tag])]); table.index.name = "set"; table.to_excel(XLSX)
    print(f"crop FD {A['fd']:.1f} | crop KD {A.get('kd_value', float('nan')):.3f} | crop FID {B['fd']:.1f} | stop sign found {det['person_detected_pct']:.1f}% | "
          f"in box IoU>=0.5 {det['person_in_box_iou50_pct']:.1f}% | mean IoU {det['mean_best_iou']:.3f} | {time.time() - t0:.0f} s -> {XLSX}")
    for d in (gen, cg, cr): shutil.rmtree(d, ignore_errors=True)
print("\nall sets done")

## The table

`crop_dinov2_fd` and `crop_dinov2_kd_value` are FD and KD between the 3,000 generated object crops and the 3,000 real crops of the same
boxes; `det_target_in_box_iou50_pct` is the share of generated images in which the detector finds a stop sign overlapping the conditioned
box with IoU ≥ 0.5, with `real_det_*` giving the same numbers on the real images. ADM has no box, so its crops are whatever happens to
be at the box location: that row is the floor of the object-centric scale.

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
t = pd.read_excel(XLSX, index_col=0)
cols = [c for c in ["crop_dinov2_fd", "crop_dinov2_kd_value", "crop_inception_fd", "crop_dinov2_precision", "crop_dinov2_recall",
                    "det_target_detected_pct", "det_target_in_box_iou50_pct", "det_mean_best_iou", "real_det_target_in_box_iou50_pct"] if c in t.columns]
print(t[cols].sort_values("crop_dinov2_fd").round(3))

## Placement test with the Traffic Signs detector

The loop above used the COCO-pretrained detector, which knows only *stop sign*. This cell re-runs the placement test with the YOLO11 model trained on the real Traffic Signs training images only (`BaselinePretrainedModel/best.pt`), any class counting as a sign, and overwrites the detector columns of the sheet. Crop FD / KD are untouched.


In [ ]:
# ==== TRAFFIC SIGNS: re-run the placement test with the Traffic Signs YOLO11 model (real images only) ======================
# The COCO-pretrained detector knows only "stop sign"; this cell replaces the detector columns of Traffic_object_metrics.xlsx with
# a detector trained on the real Traffic Signs training images. A detection of ANY class overlapping the conditioned box with
# IoU >= 0.5 counts as "sign in box". The crop FD / KD columns are untouched. Run after the CONFIG cell, same session; it
# re-stages each set from its zip (no crops, no dgm-eval), so it takes only a few minutes per set.
WEIGHTS = "/content/gdrive/MyDrive/PhD/TrafficSigns/BaselinePretrainedModel/best.pt"
CONF, IOU_THR, IMGSZ = 0.25, 0.5, 256
# ==========================================================================================================================
import os, json, shutil, zipfile, glob, time
import numpy as np, pandas as pd
from ultralytics import YOLO
assert all(n in globals() for n in ("SETS_DIR", "RESULTS_DIR", "XLSX", "REAL_INDEX")), "run the cells above (incl. CONFIG) first"
assert os.path.exists(WEIGHTS), f"weights not found: {WEIGHTS}"
det = YOLO(WEIGHTS)
print("detector classes:", det.names)

def iou_xyxy(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def detect_any(paths, boxes, batch=32):
    hit, best, found = [], [], []
    for i in range(0, len(paths), batch):
        res = det.predict(paths[i:i + batch], conf=CONF, verbose=False, imgsz=IMGSZ)   # all classes
        for box, r in zip(boxes[i:i + batch], res):
            d = r.boxes.xyxy.cpu().numpy().tolist() if r.boxes is not None and len(r.boxes) else []
            b = max((iou_xyxy(box, x) for x in d), default=0.0)
            found.append(len(d) > 0); hit.append(b >= IOU_THR); best.append(b)
    return dict(target_detected_pct=100 * float(np.mean(found)), target_in_box_iou50_pct=100 * float(np.mean(hit)), mean_best_iou=float(np.mean(best)))

table = pd.read_excel(XLSX, index_col=0)
table.index = table.index.astype(str)
real_ref = None
for tag in table.index:
    zp = f"{SETS_DIR}/{tag}.zip"
    if not os.path.exists(zp):
        print("zip missing, skipped:", tag); continue
    t0 = time.time()
    raw, gen = f"/content/dgm-eval/_raw_{tag}", f"/content/dgm-eval/gen_{tag}"
    shutil.rmtree(raw, ignore_errors=True); shutil.rmtree(gen, ignore_errors=True); os.makedirs(gen)
    with zipfile.ZipFile(zp) as z:
        z.extractall(raw)
    meta = None
    for dp, dn, fn in os.walk(raw):
        for f in fn:
            if f == "meta.json":
                meta = json.load(open(os.path.join(dp, f)))
            elif f.startswith("sample_") and f.lower().endswith(".png"):
                shutil.copy2(os.path.join(dp, f), os.path.join(gen, f))
    shutil.rmtree(raw)
    assert meta is not None, f"no meta.json in {zp}"
    samples = [s for s in meta["samples"] if os.path.exists(os.path.join(gen, s["file"]))]
    boxes = [s["bb_xyxy"] for s in samples]
    g = detect_any([os.path.join(gen, s["file"]) for s in samples], boxes)
    if real_ref is None:
        rs = [s for s in samples if s["bb_stem"] in REAL_INDEX]
        real_ref = detect_any([REAL_INDEX[s["bb_stem"]] for s in rs], [s["bb_xyxy"] for s in rs])
        print("real-image reference with this detector:", {k: round(v, 3) for k, v in real_ref.items()})
    for k, v in g.items():
        table.loc[tag, "det_" + k] = v
    for k, v in real_ref.items():
        table.loc[tag, "real_det_" + k] = v
    table.loc[tag, "detector"] = "TrafficSigns YOLO11 (real images only), any class"
    table.index.name = "set"
    table.to_excel(XLSX)
    print(f"{tag}: sign found {g['target_detected_pct']:.1f}% | in box IoU>=0.5 {g['target_in_box_iou50_pct']:.1f}% | mean IoU {g['mean_best_iou']:.3f} "
          f"({time.time() - t0:.0f} s) -> saved")
    shutil.rmtree(gen, ignore_errors=True)
pd.set_option("display.width", 220)
cols = [c for c in ["crop_dinov2_fd", "crop_dinov2_kd_value", "det_target_detected_pct", "det_target_in_box_iou50_pct", "det_mean_best_iou",
                    "real_det_target_in_box_iou50_pct"] if c in table.columns]
print(table[cols].sort_values("crop_dinov2_fd").round(3).to_string())
